In [0]:
from pyspark.sql import functions as F

storage_account_name = "staihospitaldev001"

storage_account_key = dbutils.secrets.get(
    scope="kv-aihoispital-scope",
    key="kv-aihospital-dev"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

pipeline_logs_path = gold_path + "pipeline_logs"
dq_logs_path = gold_path + "data_quality_logs"

# Read raw log delta folders
pipeline_logs = spark.read.format("delta").load(pipeline_logs_path)
dq_logs = spark.read.format("delta").load(dq_logs_path)

# Clean pipeline logs
pipeline_run_logs_gold = (
    pipeline_logs
    .withColumn(
        "pipeline_run_id",
        F.when(F.col("pipeline_run_id").isNull() | (F.col("pipeline_run_id") == ""), F.lit("manual_run"))
         .otherwise(F.col("pipeline_run_id"))
    )
    .withColumn(
        "activity_name",
        F.when(F.col("activity_name").isNull() | (F.col("activity_name") == ""), F.lit("manual_or_missing_activity"))
         .otherwise(F.col("activity_name"))
    )
    .withColumn("log_date", F.to_date("log_timestamp"))
    .withColumn("log_hour", F.hour("log_timestamp"))
)

# Clean DQ logs
data_quality_logs_gold = (
    dq_logs
    .withColumn(
        "pipeline_run_id",
        F.when(F.col("pipeline_run_id").isNull() | (F.col("pipeline_run_id") == ""), F.lit("manual_run"))
         .otherwise(F.col("pipeline_run_id"))
    )
    .withColumn(
        "activity_name",
        F.when(F.col("activity_name").isNull() | (F.col("activity_name") == ""), F.lit("manual_or_missing_activity"))
         .otherwise(F.col("activity_name"))
    )
    .withColumn("check_date", F.to_date("check_timestamp"))
    .withColumn("check_hour", F.hour("check_timestamp"))
)



display(pipeline_run_logs_gold.orderBy(F.col("log_timestamp").desc()))
display(data_quality_logs_gold.orderBy(F.col("check_timestamp").desc()))

pipeline_run_id,activity_name,stage_name,status,records_processed,message,log_timestamp,log_date,log_hour
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Succeeded,1147,"ML model training, scoring and SQL publishing completed successfully",2026-05-06T04:35:12.112494Z,2026-05-06,4
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-06T04:34:41.633892Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,Succeeded,1147,Gold fact table and patient incident feature table completed successfully,2026-05-06T04:34:07.14103Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,Started,0,Gold fact and ML feature engineering started,2026-05-06T04:33:49.097337Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_02_Silver_Data_Transformation,Silver Transformation,Succeeded,18806,"Silver admissions, incidents and staff activity completed successfully",2026-05-06T04:33:26.832798Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_02_Silver_Data_Transformation,Silver Transformation,Started,0,Silver transformation started,2026-05-06T04:32:41.363683Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_01_Bronze_Generator,Bronze Generator,Succeeded,19784,Bronze synthetic data generation completed successfully,2026-05-06T04:32:12.086292Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_01_Bronze_Generator,Bronze Generator,Started,0,Bronze synthetic data generation started,2026-05-06T04:31:21.021926Z,2026-05-06,4
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Succeeded,1147,"ML model training, scoring and SQL publishing completed successfully",2026-05-06T04:11:20.165763Z,2026-05-06,4
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-06T04:10:45.982875Z,2026-05-06,4


pipeline_run_id,activity_name,check_name,table_name,status,failed_records,message,check_timestamp,check_date,check_hour
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-06T04:36:40.444582Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Duplicate Key Check,patient_incident_predictions,Passed,0,patient_incident_predictions.AdmissionID is unique,2026-05-06T04:36:37.695363Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-06T04:36:35.01075Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PatientID has no nulls,2026-05-06T04:36:33.563748Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.AdmissionID has no nulls,2026-05-06T04:36:31.504539Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Table Not Empty,patient_incident_predictions,Passed,0,patient_incident_predictions has 1147 rows,2026-05-06T04:36:29.48849Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Duplicate Key Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.AdmissionID is unique,2026-05-06T04:36:26.727315Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.WillIncidentOccur has no nulls,2026-05-06T04:36:23.251721Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.LengthOfStayDays has no nulls,2026-05-06T04:36:19.951553Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,gold_patient_incident_features,Passed,0,gold_patient_incident_features.WardName has no nulls,2026-05-06T04:36:18.430595Z,2026-05-06,4


In [0]:
# ------------------------------------------
# AZURE SQL CONNECTION
# ------------------------------------------

sql_scope = "kv-aihospital-sql-scope"

sql_server = "aihospital-sql-dev.database.windows.net"
sql_database = "aihospital-sql-dev"
sql_username = "aihospital-sql-dev"

sql_password = dbutils.secrets.get(
    scope=sql_scope,
    key="azure-sql-password"
)

jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("Azure SQL connection loaded.")

Azure SQL connection loaded.


In [0]:
# ------------------------------------------
# FUNCTION TO WRITE TO AZURE SQL
# ------------------------------------------

def write_to_azure_sql(df, table_name, mode="overwrite"):
    df.write.jdbc(
        url=jdbc_url,
        table=table_name,
        mode=mode,
        properties=connection_properties
    )
    print(f"{table_name} written successfully to Azure SQL.")

In [0]:
# ------------------------------------------
# TEMPORARY DEMO HISTORY SEED
# Creates historical monitoring rows for Power BI trend visuals
# Remove/comment this block later if you only want live pipeline history
# ------------------------------------------

from pyspark.sql import Row
from datetime import datetime, timedelta
import random

demo_pipeline_rows = []
demo_dq_rows = []

today = datetime.utcnow().date()
start_date = today - timedelta(days=45)

pipeline_stages = [
    ("NB_01_Bronze_Generator", "Bronze Generator", 19784),
    ("NB_02_Silver_Data_Transformation", "Silver Transformation", 18806),
    ("NB_03_Gold_Feature_Engineering", "Gold Feature Engineering", 1147),
    ("NB_04_ML_Training_Scoring", "ML Training Scoring", 1147),
    ("NB_05_Data_Quality_Checks", "Data Quality Checks", 54),
    ("NB_06_Create_Monitoring_Gold_Tables", "Monitoring Gold Tables", 40)
]

failed_days = {6, 13, 22, 31, 39}

for i in range(45):
    run_date = start_date + timedelta(days=i)
    run_id = f"demo_run_{run_date.strftime('%Y%m%d')}"
    failed_run = i in failed_days

    for stage_index, (activity_name, stage_name, base_records) in enumerate(pipeline_stages):
        log_time = datetime.combine(run_date, datetime.min.time()) + timedelta(hours=6, minutes=stage_index * 12)

        # Default status
        status = "Succeeded"
        records = base_records + random.randint(-250, 250)
        message = f"{stage_name} completed successfully"

        # Introduce realistic failures
        if failed_run:
            if i in {6, 39} and stage_name == "ML Training Scoring":
                status = "Failed"
                records = 0
                message = "ML scoring failed due to model input validation issue"

            elif i in {13, 31} and stage_name == "Silver Transformation":
                status = "Failed"
                records = 0
                message = "Silver transformation failed due to source schema mismatch"

            elif i in {22} and stage_name == "Data Quality Checks":
                status = "Failed"
                records = 0
                message = "Data quality checks failed due to invalid prediction values"

        demo_pipeline_rows.append(
            Row(
                pipeline_run_id=run_id,
                activity_name=activity_name,
                stage_name=stage_name,
                status=status,
                records_processed=int(records),
                message=message,
                log_timestamp=log_time,
                log_date=run_date,
                log_hour=log_time.hour
            )
        )

    # Data quality demo checks
    dq_checks = [
        ("Table Not Empty", "silver_admissions_enriched"),
        ("Duplicate Key Check", "silver_admissions_enriched"),
        ("Null Check", "gold_patient_incident_features"),
        ("Probability Range Check", "patient_incident_predictions"),
        ("Null Check", "patient_incident_predictions")
    ]

    for check_index, (check_name, table_name) in enumerate(dq_checks):
        check_time = datetime.combine(run_date, datetime.min.time()) + timedelta(hours=7, minutes=check_index * 5)

        dq_status = "Passed"
        failed_records = 0
        dq_message = f"{check_name} passed for {table_name}"

        if i in {6, 18, 33} and table_name == "patient_incident_predictions":
            dq_status = "Failed"
            failed_records = 10 + i
            dq_message = f"{table_name} failed {check_name}: invalid or null prediction values"

        if i in {13, 31} and table_name == "gold_patient_incident_features":
            dq_status = "Failed"
            failed_records = 5 + i
            dq_message = f"{table_name} failed {check_name}: duplicate AdmissionID values found"

        demo_dq_rows.append(
            Row(
                pipeline_run_id=run_id,
                activity_name="NB_05_Data_Quality_Checks",
                check_name=check_name,
                table_name=table_name,
                status=dq_status,
                failed_records=int(failed_records),
                message=dq_message,
                check_timestamp=check_time,
                check_date=run_date,
                check_hour=check_time.hour
            )
        )

demo_pipeline_logs_df = spark.createDataFrame(demo_pipeline_rows)
demo_dq_logs_df = spark.createDataFrame(demo_dq_rows)

# Fix datatype mismatch for Delta merge
demo_pipeline_logs_df = (
    demo_pipeline_logs_df
    .withColumn("log_hour", F.col("log_hour").cast("int"))
    .withColumn("records_processed", F.col("records_processed").cast("int"))
)

demo_dq_logs_df = (
    demo_dq_logs_df
    .withColumn("check_hour", F.col("check_hour").cast("int"))
    .withColumn("failed_records", F.col("failed_records").cast("int"))
)

# Union demo history with real logs
pipeline_run_logs_gold = pipeline_run_logs_gold.unionByName(demo_pipeline_logs_df)
data_quality_logs_gold = data_quality_logs_gold.unionByName(demo_dq_logs_df)

# ------------------------------------------
# CREATE PIPELINE ACTIVITY DETAILS TABLE
# One row per pipeline run + activity
# Includes start time, end time, duration and short message
# ------------------------------------------

from pyspark.sql import functions as F

pipeline_activity_details = (
    pipeline_run_logs_gold
    .groupBy(
        "pipeline_run_id",
        "activity_name",
        "stage_name"
    )
    .agg(
        F.min("log_timestamp").alias("start_time"),
        F.max("log_timestamp").alias("end_time"),
        F.max("log_date").alias("run_date"),
        F.max("log_hour").alias("run_hour"),
        F.max("records_processed").alias("records_processed"),

        # If any activity failed, mark final status as Failed
        F.max(
            F.when(F.col("status") == "Failed", 3)
             .when(F.col("status") == "Succeeded", 2)
             .when(F.col("status") == "Started", 1)
             .otherwise(0)
        ).alias("status_rank"),

        F.max("message").alias("message")
    )
    .withColumn(
        "status",
        F.when(F.col("status_rank") == 3, F.lit("Failed"))
         .when(F.col("status_rank") == 2, F.lit("Succeeded"))
         .when(F.col("status_rank") == 1, F.lit("Started"))
         .otherwise(F.lit("Unknown"))
    )
    .withColumn(
        "duration_seconds",
        F.unix_timestamp("end_time") - F.unix_timestamp("start_time")
    )
    .withColumn(
        "duration",
        F.concat(
            F.floor(F.col("duration_seconds") / 60).cast("string"),
            F.lit("m "),
            (F.col("duration_seconds") % 60).cast("string"),
            F.lit("s")
        )
    )
        .withColumn(
            "message_summary",
            F.when(
                (F.col("status") == "Succeeded") &
                (F.col("stage_name").like("%Bronze%")),
                F.lit("Bronze ingestion completed")
            )
            .when(
                (F.col("status") == "Succeeded") &
                (F.col("stage_name").like("%Silver%")),
                F.lit("Silver transformation completed")
            )
            .when(
                (F.col("status") == "Succeeded") &
                (F.col("stage_name").like("%Gold%")),
                F.lit("Gold feature engineering completed")
            )
            .when(
                (F.col("status") == "Succeeded") &
                (F.col("stage_name").like("%ML%")),
                F.lit("ML training completed")
            )
            .when(
                F.col("status") == "Failed",
                F.lit("Pipeline activity failed")
            )
            .otherwise(F.lit("Pipeline started"))
        )
            .drop("status_rank")
        )

display(pipeline_activity_details.orderBy(F.col("start_time").desc()))

# Write curated gold monitoring tables
pipeline_run_logs_gold.write.format("delta").mode("overwrite").save(
    gold_path + "monitoring_pipeline_run_logs"
)

data_quality_logs_gold.write.format("delta").mode("overwrite").save(
    gold_path + "monitoring_data_quality_logs"
)

print("Gold monitoring tables created successfully.")

print("Temporary demo monitoring history added.")
print("Pipeline log rows:", pipeline_run_logs_gold.count())
print("Data quality log rows:", data_quality_logs_gold.count())

pipeline_run_id,activity_name,stage_name,start_time,end_time,run_date,run_hour,records_processed,message,status,duration_seconds,duration,message_summary
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-06T04:33:49.097337Z,2026-05-06T04:34:07.14103Z,2026-05-06,4,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,18,0m 18s,Gold feature engineering completed
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_02_Silver_Data_Transformation,Silver Transformation,2026-05-06T04:32:41.363683Z,2026-05-06T04:33:26.832798Z,2026-05-06,4,18806,Silver transformation started,Succeeded,45,0m 45s,Silver transformation completed
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_01_Bronze_Generator,Bronze Generator,2026-05-06T04:31:21.021926Z,2026-05-06T04:32:12.086292Z,2026-05-06,4,19784,Bronze synthetic data generation started,Succeeded,51,0m 51s,Bronze ingestion completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-06T04:09:49.151491Z,2026-05-06T04:10:09.174018Z,2026-05-06,4,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,20,0m 20s,Gold feature engineering completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_02_Silver_Data_Transformation,Silver Transformation,2026-05-06T04:08:25.756545Z,2026-05-06T04:09:16.543125Z,2026-05-06,4,18806,Silver transformation started,Succeeded,51,0m 51s,Silver transformation completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_01_Bronze_Generator,Bronze Generator,2026-05-06T04:06:48.750302Z,2026-05-06T04:07:42.293356Z,2026-05-06,4,19784,Bronze synthetic data generation started,Succeeded,54,0m 54s,Bronze ingestion completed
manual_run,manual_or_missing_activity,ML Training Scoring,2026-05-06T03:16:34.956831Z,2026-05-06T03:16:56.717545Z,2026-05-06,3,1147,ML training and scoring started,Succeeded,22,0m 22s,ML training completed
manual_run,manual_or_missing_activity,Gold Feature Engineering,2026-05-06T03:14:52.887395Z,2026-05-06T03:15:07.076707Z,2026-05-06,3,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,15,0m 15s,Gold feature engineering completed
manual_run,manual_or_missing_activity,Silver Transformation,2026-05-06T03:13:56.753357Z,2026-05-06T03:14:28.638313Z,2026-05-06,3,18806,Silver transformation started,Succeeded,32,0m 32s,Silver transformation completed
5ccdb7b8-26ff-4b48-843e-2acde239e5d8,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-06T02:59:42.086842Z,2026-05-06T02:59:57.311673Z,2026-05-06,2,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,15,0m 15s,Gold feature engineering completed


Gold monitoring tables created successfully.
Temporary demo monitoring history added.
Pipeline log rows: 324
Data quality log rows: 414


In [0]:
# ------------------------------------------
# WRITE MONITORING TABLES TO AZURE SQL
# ------------------------------------------

write_to_azure_sql(
    pipeline_run_logs_gold,
    "dbo.monitoring_pipeline_run_logs",
    mode="overwrite"
)

write_to_azure_sql(
    data_quality_logs_gold,
    "dbo.monitoring_data_quality_logs",
    mode="overwrite"
)
write_to_azure_sql(
    pipeline_activity_details,
    "dbo.monitoring_pipeline_activity_details",
    mode="overwrite"
)

dbo.monitoring_pipeline_run_logs written successfully to Azure SQL.
dbo.monitoring_data_quality_logs written successfully to Azure SQL.
dbo.monitoring_pipeline_activity_details written successfully to Azure SQL.


In [0]:
# ------------------------------------------
# VALIDATE SQL TABLES
# ------------------------------------------

pipeline_sql_check = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.monitoring_pipeline_run_logs",
    properties=connection_properties
)

dq_sql_check = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.monitoring_data_quality_logs",
    properties=connection_properties
)

pipeline_activity_check = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.monitoring_pipeline_activity_details",
    properties=connection_properties
)

print("SQL monitoring_pipeline_run_logs rows:", pipeline_sql_check.count())
print("SQL monitoring_data_quality_logs rows:", dq_sql_check.count())
print("SQL monitoring_pipeline_activity_details:", pipeline_activity_check.count())

display(pipeline_sql_check)
display(dq_sql_check)
display(pipeline_activity_check)

SQL monitoring_pipeline_run_logs rows: 324
SQL monitoring_data_quality_logs rows: 414
SQL monitoring_pipeline_activity_details: 292


pipeline_run_id,activity_name,stage_name,status,records_processed,message,log_timestamp,log_date,log_hour
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-04T10:16:13.813Z,2026-05-04,10
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-04T11:02:22.52Z,2026-05-04,11
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-06T04:10:45.983Z,2026-05-06,4
manual_run,NB_04_ML_Training_Scoring,ML Training Scoring,Started,0,ML training and scoring started,2026-05-06T04:34:41.633Z,2026-05-06,4
manual_run,manual_or_missing_activity,Bronze Generator,Succeeded,19784,Bronze synthetic data generation completed successfully,2026-05-04T09:54:35.353Z,2026-05-04,9
manual_run,manual_or_missing_activity,Bronze Generator,Succeeded,19784,Bronze synthetic data generation completed successfully,2026-05-06T03:12:50.567Z,2026-05-06,3
manual_run,manual_or_missing_activity,Gold Feature Engineering,Started,0,Gold fact and ML feature engineering started,2026-05-06T03:14:52.887Z,2026-05-06,3
manual_run,manual_or_missing_activity,Bronze Generator,Started,0,Bronze synthetic data generation started,2026-05-04T09:54:00.953Z,2026-05-04,9
manual_run,manual_or_missing_activity,Bronze Generator,Started,0,Bronze synthetic data generation started,2026-05-04T09:52:36.087Z,2026-05-04,9
manual_run,manual_or_missing_activity,Bronze Generator,Started,0,Bronze synthetic data generation started,2026-05-06T03:12:33.963Z,2026-05-06,3


pipeline_run_id,activity_name,check_name,table_name,status,failed_records,message,check_timestamp,check_date,check_hour
83cc6659-dc34-4dc9-81f6-72a49d555295,NB_05_Data_Quality_Checks,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-04T11:04:20.567Z,2026-05-04,11
83cc6659-dc34-4dc9-81f6-72a49d555295,NB_05_Data_Quality_Checks,Table Not Empty,patient_incident_predictions,Passed,0,patient_incident_predictions has 1147 rows,2026-05-04T11:04:09.523Z,2026-05-04,11
5ccdb7b8-26ff-4b48-843e-2acde239e5d8,NB_05_Data_Quality_Checks,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-06T03:02:40.547Z,2026-05-06,3
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_05_Data_Quality_Checks,Null Check,silver_admissions_enriched,Passed,0,silver_admissions_enriched.PatientID has no nulls,2026-05-06T04:12:22.587Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-06T04:36:40.443Z,2026-05-06,4
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_05_Data_Quality_Checks,Probability Range Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability values are between 0 and 1,2026-05-06T04:13:04.517Z,2026-05-06,4
83cc6659-dc34-4dc9-81f6-72a49d555295,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-04T11:04:15.243Z,2026-05-04,11
5ccdb7b8-26ff-4b48-843e-2acde239e5d8,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-06T03:02:36.093Z,2026-05-06,3
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-06T04:12:59.657Z,2026-05-06,4
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_05_Data_Quality_Checks,Null Check,patient_incident_predictions,Passed,0,patient_incident_predictions.PredictedIncidentProbability has no nulls,2026-05-06T04:36:35.01Z,2026-05-06,4


pipeline_run_id,activity_name,stage_name,start_time,end_time,run_date,run_hour,records_processed,message,status,duration_seconds,duration,message_summary
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_01_Bronze_Generator,Bronze Generator,2026-05-06T04:31:21.023Z,2026-05-06T04:32:12.087Z,2026-05-06,4,19784,Bronze synthetic data generation started,Succeeded,51,0m 51s,Bronze ingestion completed
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_02_Silver_Data_Transformation,Silver Transformation,2026-05-06T04:32:41.363Z,2026-05-06T04:33:26.833Z,2026-05-06,4,18806,Silver transformation started,Succeeded,45,0m 45s,Silver transformation completed
01da6dc5-357a-47d5-81cb-0b028dae3fce,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-06T04:33:49.097Z,2026-05-06T04:34:07.14Z,2026-05-06,4,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,18,0m 18s,Gold feature engineering completed
49cc30f6-a87a-44be-930f-d8422577aa52,NB_01_Bronze_Generator,Bronze Generator,2026-05-04T10:13:13.897Z,2026-05-04T10:14:01.75Z,2026-05-04,10,19784,Bronze synthetic data generation started,Succeeded,48,0m 48s,Bronze ingestion completed
49cc30f6-a87a-44be-930f-d8422577aa52,NB_02_Silver_Data_Transformation,Silver Transformation,2026-05-04T10:14:30.847Z,2026-05-04T10:15:16.947Z,2026-05-04,10,18806,Silver transformation started,Succeeded,46,0m 46s,Silver transformation completed
49cc30f6-a87a-44be-930f-d8422577aa52,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-04T10:15:37.117Z,2026-05-04T10:15:55.85Z,2026-05-04,10,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,18,0m 18s,Gold feature engineering completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_01_Bronze_Generator,Bronze Generator,2026-05-06T04:06:48.75Z,2026-05-06T04:07:42.293Z,2026-05-06,4,19784,Bronze synthetic data generation started,Succeeded,54,0m 54s,Bronze ingestion completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_02_Silver_Data_Transformation,Silver Transformation,2026-05-06T04:08:25.757Z,2026-05-06T04:09:16.543Z,2026-05-06,4,18806,Silver transformation started,Succeeded,51,0m 51s,Silver transformation completed
5508323f-bb1b-4dd4-ac07-37f9a9bb07b8,NB_03_Gold_Feature_Engineering,Gold Feature Engineering,2026-05-06T04:09:49.15Z,2026-05-06T04:10:09.173Z,2026-05-06,4,1147,Gold fact table and patient incident feature table completed successfully,Succeeded,20,0m 20s,Gold feature engineering completed
5ccdb7b8-26ff-4b48-843e-2acde239e5d8,NB_01_Bronze_Generator,Bronze Generator,2026-05-06T02:57:50.68Z,2026-05-06T02:58:06.097Z,2026-05-06,2,19784,Bronze synthetic data generation started,Succeeded,16,0m 16s,Bronze ingestion completed
